# Exploratory Data Analysis: Credit Default Risk

**Business question:** which applicant characteristics are associated with loan default, and how severe is the class imbalance we need to design around?

Run `python data/generate_synthetic_data.py` first to produce `data/credit_data.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/credit_data.csv')
df.head()

## 1. Class balance
This is the single most important fact about the dataset -- it shapes every modeling decision downstream (metric choice, resampling, threshold selection).

In [ ]:
default_rate = df['default'].mean()
print(f"Default rate: {default_rate:.2%}")
print(f"Class ratio (good:bad): {(1-default_rate)/default_rate:.1f}:1")

df['default'].value_counts(normalize=True).plot(kind='bar', title='Class balance')
plt.xticks([0, 1], ['No default', 'Default'], rotation=0)
plt.show()

## 2. Univariate risk relationships
Sanity-check that the features behave the way domain knowledge would predict -- if FICO score *doesn't* separate defaulters from non-defaulters, something's wrong with the data before we even get to modeling.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, col in zip(axes.flat, ['fico_score', 'dti', 'revol_util', 'annual_income']):
    sns.kdeplot(data=df, x=col, hue='default', ax=ax, common_norm=False)
    ax.set_title(f'{col} by default status')

plt.tight_layout()
plt.show()

## 3. Correlation structure
Check for multicollinearity before feeding features into logistic regression -- it doesn't hurt tree models but it does distort the interpretability of linear coefficients.

In [ ]:
numeric_cols = ['annual_income', 'loan_amnt', 'int_rate', 'dti', 'revol_util',
                 'open_acc', 'delinq_2yrs', 'fico_score', 'emp_length', 'default']
corr = df[numeric_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature correlation matrix')
plt.show()

## 4. Categorical features vs. default rate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.groupby('home_ownership')['default'].mean().plot(kind='bar', ax=axes[0], title='Default rate by home ownership')
df.groupby('purpose')['default'].mean().plot(kind='bar', ax=axes[1], title='Default rate by loan purpose')
plt.tight_layout()
plt.show()

## Key takeaways to carry into modeling

1. **~3-6% default rate** -- this is a rare-event classification problem. Accuracy is a useless metric here (predicting "no default" for everyone would already score >94%). Use ROC-AUC, PR-AUC, and business cost as evaluation metrics instead.
2. **FICO, DTI, and revolving utilization** show the clearest separation between classes -- expect these to dominate feature importance.
3. **Class imbalance** means the model needs `class_weight='balanced'` or resampling, and predictions need probability calibration, not just a 0.5 threshold.

Next: `src/train.py` for the modeling pipeline.